# Violencia en Tech — Notebook 01: Extracción de Datos de Reddit

**Proyecto:** Sistema de Análisis de Violencia Simbólica contra Mujeres en Tecnología  
**Autora:** Maricarmen Camacho Pérez — UAEH, Ciencias Computacionales  
**Input:** Credenciales de Reddit API (`config/credentials.py`)  
**Output:** `data/raw/reddit_data_raw_<timestamp>.csv`  

---
**Pipeline:** **01_Extraccion** → 02_EDA → 03_Limpieza → 04_Reglas → 05_BERT → 06_Temporal → Final

---

### ¿Qué hace este notebook?
1. Se conecta a la API de Reddit usando credenciales seguras
2. Define las 17 comunidades (subreddits) a analizar
3. Define las 58 palabras clave para buscar contenido relevante
4. Busca cada palabra clave en cada subreddit
5. Extrae los posts encontrados y sus comentarios
6. Guarda todo en un CSV para el siguiente notebook

---
## 1. Instalación e importación de librerías

In [28]:
# ══════════════════════════════════════════════════════════
# INSTALACIÓN DE DEPENDENCIAS
# Se instalan con --quiet para que no llene la pantalla de texto
# ══════════════════════════════════════════════════════════

# praw       → librería oficial para conectarse a la API de Reddit
# prawcore   → manejo de errores de conexión con Reddit (rate limits, etc.)
# pandas     → para crear y manipular tablas de datos (DataFrames)
# tqdm       → muestra barras de progreso en los ciclos largos
%pip install praw prawcore pandas tqdm --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\marii\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [29]:
# ══════════════════════════════════════════════════════════
# IMPORTACIÓN DE LIBRERÍAS
# Cada una tiene un propósito específico en este notebook
# ══════════════════════════════════════════════════════════

import praw              # Para conectarse a Reddit y hacer búsquedas de posts
import prawcore          # Para detectar errores como "demasiadas peticiones" (rate limit)
import pandas as pd      # Para crear tablas (DataFrames) con los datos extraídos
from datetime import datetime, timezone  # Para manejar fechas en formato UTC (hora universal)
import time              # Para poner pausas entre peticiones y no saturar la API
from tqdm import tqdm    # Para mostrar barras de progreso al recorrer las keywords
import sys               # Para acceder a información del sistema (versión de Python, rutas)
import os                # Para manejar rutas de archivos y carpetas del proyecto
import warnings          # Para ocultar mensajes de advertencia que no afectan la ejecución

warnings.filterwarnings('ignore')  # Desactivar warnings para que la salida se vea limpia

# Agregar la carpeta raíz del proyecto al path de Python
# Sin esta línea, Python no puede encontrar el archivo config/credentials.py
# porque estamos ejecutando desde la carpeta notebooks/ y config/ está un nivel arriba
sys.path.insert(0, os.path.abspath('..'))  # '..' significa "la carpeta de arriba"

# Imprimir información del entorno para documentar con qué versiones se ejecutó
print(f"Fecha de ejecución : {datetime.now().strftime('%Y-%m-%d %H:%M')}")  # Cuándo se corrió
print(f"Python             : {sys.version.split()[0]}")  # Versión de Python
print(f"Pandas             : {pd.__version__}")  # Versión de pandas

Fecha de ejecución : 2026-03-18 14:58
Python             : 3.11.9
Pandas             : 3.0.1


---
## 2. Conexión a Reddit API

Las credenciales se cargan desde `config/credentials.py` (protegido por `.gitignore`).  
Eso significa que el archivo **nunca se sube a GitHub** y tus claves están seguras.  

Si no tienes ese archivo:

1. Llena tus datos de Reddit API en config/credentials.py
2. Obtén tus credenciales en: https://www.reddit.com/prefs/apps

In [30]:
# ══════════════════════════════════════════════════════════
# CARGAR CREDENCIALES DE FORMA SEGURA
# Las credenciales están en un archivo aparte (config/credentials.py)
# 
# ══════════════════════════════════════════════════════════

try:
    # Intentar importar las 3 variables del archivo de credenciales
    from config.credentials import REDDIT_CLIENT_ID       # ID de tu app en Reddit
    from config.credentials import REDDIT_CLIENT_SECRET   # Clave secreta de tu app
    from config.credentials import REDDIT_USER_AGENT      # Nombre que identifica tu app
    print("Credenciales cargadas desde config/credentials.py")

except ImportError:
    # Si el archivo no existe, mostrar instrucciones claras de qué hacer
    raise FileNotFoundError(
        "\n No se encontró config/credentials.py\n"
        "\n Pasos para solucionarlo:\n"
        "   1. Copia config/credentials_template.py → config/credentials.py\n"
        "   2. Abre config/credentials.py y llena tus datos\n"
        "   3. Obtén credenciales en: https://www.reddit.com/prefs/apps\n"
    )

Credenciales cargadas desde config/credentials.py


In [31]:
# ══════════════════════════════════════════════════════════
# CREAR LA CONEXIÓN CON REDDIT
# praw.Reddit() crea un objeto que permite buscar posts y comentarios.
# Se usa modo "solo lectura" (no necesitamos publicar, solo leer).
# ══════════════════════════════════════════════════════════

reddit = praw.Reddit(
    client_id=REDDIT_CLIENT_ID,         # El ID que te da Reddit al crear tu app
    client_secret=REDDIT_CLIENT_SECRET,  # La clave secreta de tu app
    user_agent=REDDIT_USER_AGENT         # Un nombre para identificarte ante Reddit
)

# reddit.read_only debe ser True → solo leemos datos, no publicamos nada
print(f"Modo solo lectura : {reddit.read_only}")  # Debe imprimir True
print("Conexión a Reddit exitosa.")

Modo solo lectura : True
Conexión a Reddit exitosa.


---
## 3. Parámetros del estudio

Se definen los tres elementos principales:
- **Período:** de cuándo a cuándo extraemos datos (2016–2026)
- **Comunidades:** en qué 17 subreddits buscamos
- **Palabras clave:** qué 45 términos buscamos dentro de esos subreddits

In [32]:
# ══════════════════════════════════════════════════════════
# PERÍODO DE ANÁLISIS
# Definimos la ventana temporal de los datos que queremos.
# tzinfo=timezone.utc indica que las fechas están en hora universal (UTC)
# que es el estándar que usa Reddit internamente para sus timestamps.
# ══════════════════════════════════════════════════════════

FECHA_INICIO = datetime(2016, 1, 1, tzinfo=timezone.utc)   # Desde el 1 de enero de 2016
FECHA_FIN    = datetime(2026, 1, 31, tzinfo=timezone.utc)   # Hasta el 31 de enero de 2026

# Calcular cuántos días abarca el período
dias_totales = (FECHA_FIN - FECHA_INICIO).days  # Resta de fechas da los días

print(f"Período de extracción:")
print(f"  Inicio   : {FECHA_INICIO.strftime('%d de %B de %Y')}")  # Formato legible
print(f"  Fin      : {FECHA_FIN.strftime('%d de %B de %Y')}")
print(f"  Duración : {dias_totales} días (~{dias_totales // 365} años)")

Período de extracción:
  Inicio   : 01 de January de 2016
  Fin      : 31 de January de 2026
  Duración : 3683 días (~10 años)


In [33]:
# ══════════════════════════════════════════════════════════
# COMUNIDADES (SUBREDDITS) A ANALIZAR
# Se organizan en un diccionario por tipo de comunidad.
# Esta clasificación es la misma que aparece en el artículo (sección 3.1).
#
# ¿Por qué se organizan así?
# Porque en el artículo se justifica la selección de comunidades
# en tres grupos según su función y tipo de participantes.
# ══════════════════════════════════════════════════════════

SUBREDDITS = {

    # ── Grupo 1: Comunidades de apoyo para mujeres ───────
    # Espacios creados específicamente para mujeres en STEM.
    # Aquí las mujeres comparten experiencias y se apoyan entre sí.
    "apoyo_mujeres": [
        "TwoXChromosomes",    # Comunidad feminista general, +13M miembros
        "girlsgonewired",     # Mujeres en tecnología e informática
        "WomenInTech",        # Experiencias de mujeres en la industria tech
        "LadiesofScience",    # Mujeres en ciencia, tecnología, ingeniería y matemáticas
        "WomenEngineers",     # Mujeres en ingeniería
        
    ],

    # ── Grupo 2: Comunidades profesionales mixtas ────────
    # Foros de carrera y tecnología donde participan todos los géneros.
    # Aquí es donde más se encuentra la violencia sutil (mansplaining, etc.).
    "profesional_mixta": [
        "cscareerquestions",  # Preguntas sobre carreras en Ciencias Computacionales
        "ITCareerQuestions",  # Preguntas sobre carreras en IT (soporte, redes, etc.)
        "experienceddevs",    # Desarrolladores con experiencia, cultura laboral
        "AskComputerScience", # Preguntas académicas sobre ciencias de la computación
        "programming",        # Discusiones generales sobre programación
        "learnprogramming",   # Comunidad de aprendizaje, incluye principiantes
        "webdev",             # Desarrollo web (frontend, backend, fullstack)
        "datascience",        # Ciencia de datos, estadística y análisis
        "MachineLearning",    # Inteligencia artificial y aprendizaje automático
    ],

    # ── Grupo 3: Comunidades laborales generales ─────────
    # Espacios amplios donde se habla de condiciones de trabajo
    # y cultura tecnológica en general.
    "laboral_general": [
        "technology",         # Noticias y cultura tecnológica
        "antiwork",           # Discusiones sobre condiciones laborales abusivas
        "AskWomen",          # Preguntas dirigidas a mujeres sobre cualquier tema, incluyendo trabajo
    ]
}

# Crear una lista plana con todos los subreddits
# El ciclo recorre el diccionario y junta todos los nombres en una sola lista
# Ejemplo: ["TwoXChromosomes", "girlsgonewired", ..., "antiwork", "womenintech"]
ALL_SUBREDDITS = [sub for grupo in SUBREDDITS.values() for sub in grupo]

# Mostrar resumen
print(f"Total de subreddits: {len(ALL_SUBREDDITS)}")
print(f"  Apoyo mujeres    : {len(SUBREDDITS['apoyo_mujeres'])} comunidades")
print(f"  Profesional mixta: {len(SUBREDDITS['profesional_mixta'])} comunidades")
print(f"  Laboral general  : {len(SUBREDDITS['laboral_general'])} comunidades")
print()
# Listar cada subreddit con su número
for i, sub in enumerate(ALL_SUBREDDITS, 1):  # enumerate empieza a contar desde 1
    print(f"  {i:2}. r/{sub}")                # :2 hace que ocupe 2 espacios (alinea)

Total de subreddits: 17
  Apoyo mujeres    : 5 comunidades
  Profesional mixta: 9 comunidades
  Laboral general  : 3 comunidades

   1. r/TwoXChromosomes
   2. r/girlsgonewired
   3. r/WomenInTech
   4. r/LadiesofScience
   5. r/WomenEngineers
   6. r/cscareerquestions
   7. r/ITCareerQuestions
   8. r/experienceddevs
   9. r/AskComputerScience
  10. r/programming
  11. r/learnprogramming
  12. r/webdev
  13. r/datascience
  14. r/MachineLearning
  15. r/technology
  16. r/antiwork
  17. r/AskWomen


In [34]:
# ══════════════════════════════════════════════════════════
# PALABRAS CLAVE (KEYWORDS) PARA LA BÚSQUEDA
# ══════════════════════════════════════════════════════════
#
# Las keywords cumplen DOS funciones en este proyecto:
#   1. Guían la búsqueda en Reddit para recolectar textos relevantes
#   2. Determinan la composición del corpus (qué tipo de contenido entra)
#
# CRITERIO DE SELECCIÓN:
#   - Cada keyword debe ser lo suficientemente ESPECÍFICA para traer
#     contenido relevante, pero no tan genérica que traiga ruido.
#   - Se eliminaron keywords genéricas como "stay at home", "well actually"
#     o "being difficult" porque generan demasiados falsos positivos
#     (textos que no tienen nada que ver con violencia de género).
#   - Se agregaron términos documentados en la literatura sobre
#     discriminación de género en tecnología (Sue, 2010; Benítez-Eyzaguirre, 2021).
#   - Se incluyó una categoría de "contexto neutro" para que el corpus
#     también tenga textos sobre mujeres en tech SIN violencia,
#     lo que ayuda a BERT a aprender qué es "neutral".
#
# IMPORTANTE: Las keywords solo guían la búsqueda.
# La etiqueta real (directa/sutil/neutral) la asigna el clasificador
# BERT en el Notebook 05. Un texto encontrado con una keyword de
# "violencia_directa" puede terminar clasificado como neutral.
# ══════════════════════════════════════════════════════════

KEYWORDS = {

    # ── Violencia directa ────────────────────────────────
    # Insultos, exclusión explícita, acoso abierto, discriminación evidente.
    # Criterio: expresiones que la mayoría reconocería como agresivas
    # o discriminatorias sin necesidad de contexto adicional.
    "violencia_directa": [
        # --- Descalificación profesional explícita ---
        "women cant code",                # Negación directa de capacidad técnica
        "girls cant code",                # Variante con "girls"
        "women dont belong in tech",      # Exclusión explícita del sector
        "girls dont belong in tech",      # Variante con "girls"
        "not technical enough woman",     # Cuestionamiento abierto de competencia
        "too emotional for programming",  # Estereotipo emocional como descalificación
        "women ruin tech",                # Culpar a las mujeres del deterioro del sector

        # --- Etiquetas despectivas de contratación ---
        "diversity hire",                 # Reducir logros a una cuota (muy documentado)
        "affirmative action hire",        # Variante: contratación por acción afirmativa
        "token woman",                    # "La mujer de adorno" (cuota de género)
        "hiring quota woman",             # Otra variante de cuota

        # --- Acoso y ambiente hostil ---
        "sexual harassment tech",         # Acoso sexual en el sector tech
        "hostile work environment",       # Ambiente laboral hostil (término legal en EE.UU.)
        "toxic tech workplace",           # Ambiente tóxico en tech
        "threatening behavior at work",   # Comportamiento amenazante laboral
        "sexist boss tech",               # Jefe sexista en tech
    ],

    # ── Violencia sutil ──────────────────────────────────
    # Mansplaining, microagresiones, condescendencia, gaslighting,
    # exclusión implícita, cuestionamiento velado de competencia.
    # Criterio: expresiones que muchas personas no reconocen como
    # violencia pero que la literatura documenta como dañinas
    # (Sue, 2010; Benítez-Eyzaguirre, 2021).
    "violencia_sutil": [
        # --- Mansplaining y condescendencia ---
        "mansplaining",                   # Término específico, baja ambigüedad
        "mansplained to me",              # Variante en primera persona (testimonial)
        "condescending coworker",         # Compañero condescendiente
        "talked down to me",              # "Me habló con condescendencia"
        "questioned my competence",       # Cuestionamiento directo de capacidad

        # --- Microagresiones documentadas ---
        "microaggression",                # Término académico específico
        "good for a woman",               # Halago con sesgo ("para ser mujer")
        "surprisingly good for a girl",   # Variante: sorpresa ante competencia femenina
        "impressive for a woman",         # Variante: admiración condicionada al género
        "you dont look like programmer",  # Estereotipo de apariencia vs. profesión

        # --- Invalidación y gaslighting ---
        "gaslighting at work",            # Manipulación psicológica laboral
        "too sensitive at work",          # Invalidación con contexto laboral (evita ruido)
        "overreacting at work",           # "Exageras" en contexto laboral
        "cant take a joke sexist",        # "No aguantas bromas" + filtro de contexto

        # --- Cultura excluyente ---
        "boys club tech",                 # "Club de chicos" en tech
        "bro culture programming",        # Cultura "brogrammer"
        "old boys network tech",          # Red de contactos solo masculina
        "glass ceiling tech",             # Techo de cristal en tech
    ],

    # ── Experiencias y testimonios ───────────────────────
    # Mujeres relatando vivencias en el sector tech.
    # Criterio: frases que señalan narrativa en primera persona
    # o descripción de experiencias de género en el sector.
    "experiencia": [
        # --- Relatos en primera persona ---
        "as a woman in tech",             # Inicio clásico de testimonio
        "my experience as female developer", # Experiencia como desarrolladora
        "female engineer experience",     # Experiencia de ingeniera
        "being the only woman",           # Ser la única mujer (muy frecuente)
        "only woman on the team",         # Variante: única en el equipo
        "only female engineer",           # Variante: única ingeniera

        # --- Síntomas y consecuencias ---
        "imposter syndrome tech",         # Síndrome del impostor con contexto tech
        "feeling excluded at work",       # Exclusión con contexto laboral
        "not taken seriously at work",    # No ser tomada en serio (laboral)
        "left tech because",              # Mujeres que abandonaron el sector

        # --- Discriminación estructural ---
        "gender discrimination workplace",# Discriminación de género laboral
        "sexism in programming",          # Sexismo en programación
        "bias against women tech",        # Sesgo contra mujeres en tech
        "women in stem challenges",       # Retos de mujeres en STEM
        "pay gap woman tech",             # Brecha salarial de género en tech
        "gender pay gap software",        # Variante: brecha salarial en software
    ],

    # ── Contexto neutro (NUEVA CATEGORÍA) ────────────────
    # Textos sobre mujeres en tech que probablemente NO contengan violencia.
    # ¿Por qué incluir esto? Porque si TODO el corpus viene de búsquedas
    # sobre violencia, BERT no tiene buenos ejemplos de qué es "neutral".
    # Estos términos traen contenido positivo o informativo que equilibra el corpus.
    "contexto_neutro": [
        "women in tech success",          # Historias de éxito
        "female developer advice",        # Consejos para desarrolladoras
        "women coding bootcamp",          # Bootcamps para mujeres
        "career advice woman tech",       # Consejos de carrera
        "mentoring women engineering",    # Mentoría para mujeres en ingeniería
        "women tech conference",          # Conferencias de mujeres en tech
        "girls who code",                 # Programa educativo conocido
        "women hackathon",                # Hackathons para mujeres
    ]
}

# Combinar todas las keywords en una sola lista plana para el ciclo de extracción
ALL_KEYWORDS = [kw for grupo in KEYWORDS.values() for kw in grupo]

# Mostrar resumen
print(f"Total de palabras clave: {len(ALL_KEYWORDS)}")
print()
for categoria, kws in KEYWORDS.items():
    print(f"  {categoria}: {len(kws)} keywords")

Total de palabras clave: 58

  violencia_directa: 16 keywords
  violencia_sutil: 18 keywords
  experiencia: 16 keywords
  contexto_neutro: 8 keywords


---
## 4. Funciones de extracción

Estas funciones convierten los objetos de Reddit (Post, Comment) en  
diccionarios de Python con los campos que necesitamos para el análisis.

In [35]:
def fecha_en_rango(timestamp_utc):
    """
    Verifica si una fecha cae dentro del período 2016–2026.
    
    ¿Por qué existe esta función?
    Porque la API de Reddit NO permite filtrar por fecha en las búsquedas.
    Reddit devuelve resultados de cualquier fecha, y nosotros filtramos después.
    
    Parámetros:
        timestamp_utc (float): fecha en formato Unix (segundos desde 1970-01-01)
    Retorna:
        bool: True si la fecha está en el rango, False si no
    """
    # Convertir el número (timestamp) a un objeto datetime que Python entiende
    # Ejemplo: 1672531200 → datetime(2023, 1, 1, 0, 0, tzinfo=UTC)
    fecha = datetime.fromtimestamp(timestamp_utc, tz=timezone.utc)
    
    # Retornar True solo si la fecha está entre FECHA_INICIO y FECHA_FIN
    return FECHA_INICIO <= fecha <= FECHA_FIN


def extraer_post(post, keyword):
    """
    Convierte un objeto Post de Reddit en un diccionario con los campos
    que necesitamos para el análisis.
    
    Un "post" es una publicación original (el inicio de un hilo).
    Tiene título + texto. Los comentarios son las respuestas al post.
    
    Parámetros:
        post: objeto Post de PRAW (viene de la API de Reddit)
        keyword (str): la palabra clave que encontró este post
    Retorna:
        dict con los datos, o None si ocurre un error
    """
    try:
        # Convertir timestamp a fecha UNA SOLA VEZ (antes se hacía 3 veces, desperdicio)
        fecha = datetime.fromtimestamp(post.created_utc, tz=timezone.utc)
        
        return {
            # ── Identificación ───────────────────────────
            "id":              post.id,                          # ID único en Reddit (ej: "abc123")
            "tipo":            "post",                           # Marcar que es post (no comentario)
            "subreddit":       post.subreddit.display_name,      # Nombre del subreddit (ej: "WomenInTech")
            
            # ── Contenido ────────────────────────────────
            "titulo":          post.title,                       # Título del post
            "texto":           post.selftext or "",              # Cuerpo del post ("" si es un link-post)
            "texto_completo":  f"{post.title} {post.selftext or ''}".strip(),  # Título + texto juntos
            # ↑ Se juntan porque el clasificador BERT necesita el contexto completo
            
            # ── Metadatos ────────────────────────────────
            "autor":           str(post.author) if post.author else "[deleted]",  # Autor o [deleted]
            "fecha":           fecha,                            # Fecha como datetime
            "año":             fecha.year,                       # Año (para agrupar por período)
            "mes":             fecha.month,                      # Mes
            "año_mes":         fecha.strftime("%Y-%m"),           # "2024-03" (para timeline mensual)
            "upvotes":         post.score,                       # Puntuación (upvotes - downvotes)
            "num_comentarios": post.num_comments,                # Cuántos comentarios tiene
            "url":             f"https://reddit.com{post.permalink}",  # Link directo al post
            "keyword":         keyword,                          # Qué keyword lo encontró
        }
    except Exception:
        # Si algo falla (post borrado, error de red, campo faltante, etc.)
        # retornamos None y el ciclo principal lo ignora
        return None


def extraer_comentario(comment, subreddit_name, keyword):
    """
    Convierte un objeto Comment de Reddit en un diccionario.
    
    Los comentarios son clave porque ahí es donde ocurre la mayor
    parte de la interacción: tanto la violencia conversacional
    como los testimonios de experiencias vividas.
    El 92.6% de nuestro corpus final son comentarios.
    
    Parámetros:
        comment: objeto Comment de PRAW
        subreddit_name (str): nombre del subreddit
        keyword (str): la keyword que encontró el post padre
    Retorna:
        dict con los datos, o None si ocurre un error
    """
    try:
        # Convertir timestamp a fecha (una sola vez)
        fecha = datetime.fromtimestamp(comment.created_utc, tz=timezone.utc)
        
        return {
            # ── Identificación ───────────────────────────
            "id":              comment.id,                       # ID único del comentario
            "tipo":            "comment",                        # Marcar que es comentario
            "subreddit":       subreddit_name,                   # Comunidad de origen
            
            # ── Contenido ────────────────────────────────
            "titulo":          "",                               # Los comentarios no tienen título
            "texto":           comment.body,                     # Texto del comentario
            "texto_completo":  comment.body,                     # Mismo (no hay título que concatenar)
            
            # ── Metadatos ────────────────────────────────
            "autor":           str(comment.author) if comment.author else "[deleted]",
            "fecha":           fecha,
            "año":             fecha.year,
            "mes":             fecha.month,
            "año_mes":         fecha.strftime("%Y-%m"),
            "upvotes":         comment.score,
            "num_comentarios": None,                             # Comentarios no tienen este campo
            "url":             f"https://reddit.com{comment.permalink}",
            "keyword":         keyword,
        }
    except Exception:
        return None


print("Funciones de extracción definidas.")

Funciones de extracción definidas.


---
## 5. Extracción principal

Este es el ciclo más importante. Funciona así:

```
Para cada subreddit (17 comunidades):
    Para cada keyword (58 palabras clave):
        → Buscar posts que contengan esa keyword
        → Para cada post encontrado:
            → ¿Está dentro del período 2016-2026? Si no, saltar
            → ¿Ya lo extrajimos antes? Si sí, saltar
            → Guardar datos del post
            → Extraer hasta 20 comentarios de ese post
        → Pausar 0.5 segundos (respetar rate limit)
    → Guardar checkpoint cada 3 subreddits
```

**Duración estimada:** varias horas (depende de la velocidad de la API).

In [36]:
# ══════════════════════════════════════════════════════════
# CONSTANTES DE CONFIGURACIÓN
# Se definen aquí arriba para ajustarlas fácilmente si es necesario
# ══════════════════════════════════════════════════════════

MAX_POSTS_POR_BUSQUEDA = 50    # Máximo que devuelve la API sin paginación
MAX_COMENTARIOS_POR_POST = 20  # Balance entre cobertura y velocidad
PAUSA_ENTRE_BUSQUEDAS = 0.5    # Segundos entre requests (Reddit permite 60 req/min)
CHECKPOINT_CADA_N_SUBS = 3     # Cada cuántos subreddits guardar respaldo

# ══════════════════════════════════════════════════════════
# VARIABLES DE CONTROL
# ══════════════════════════════════════════════════════════

datos = []          # Lista donde se acumulan TODOS los registros extraídos
ids_vistos = set()  # Conjunto de IDs ya procesados (set es más rápido que list para buscar)
errores = []        # Lista de errores que ocurran durante la extracción
t_inicio = time.time()  # Momento de inicio (para calcular duración al final)

print(f"Configuración:")
print(f"  Posts por búsqueda   : {MAX_POSTS_POR_BUSQUEDA}")
print(f"  Comentarios por post : {MAX_COMENTARIOS_POR_POST}")
print(f"  Pausa entre búsquedas: {PAUSA_ENTRE_BUSQUEDAS}s")
print(f"  Checkpoint cada      : {CHECKPOINT_CADA_N_SUBS} subreddits")
print(f"\nIniciando extracción de {len(ALL_SUBREDDITS)} subreddits × {len(ALL_KEYWORDS)} keywords...")

Configuración:
  Posts por búsqueda   : 50
  Comentarios por post : 20
  Pausa entre búsquedas: 0.5s
  Checkpoint cada      : 3 subreddits

Iniciando extracción de 17 subreddits × 58 keywords...


In [37]:
# ══════════════════════════════════════════════════════════
# CICLO PRINCIPAL DE EXTRACCIÓN
# ══════════════════════════════════════════════════════════

# Recorrer cada subreddit (enumerate cuenta desde 1 para mostrar progreso)
for idx, sub_name in enumerate(ALL_SUBREDDITS, 1):
    
    # Imprimir qué subreddit estamos procesando
    print(f"\n[{idx}/{len(ALL_SUBREDDITS)}] r/{sub_name}")

    # ── PASO 1: Verificar que el subreddit existe y es accesible ──
    # Algunos subreddits pueden estar privados, baneados o no existir
    try:
        subreddit = reddit.subreddit(sub_name)  # Crear referencia al subreddit
        next(subreddit.hot(limit=1))            # Pedir 1 post para verificar acceso
    except Exception as e:
        # Si no se puede acceder, lo saltamos y seguimos con el siguiente
        print(f"  SKIP — No accesible: {e}")
        errores.append({"subreddit": sub_name, "error": str(e)})
        continue  # "continue" salta al siguiente subreddit

    count = 0  # Contador de textos extraídos de ESTE subreddit

    # ── PASO 2: Buscar cada keyword dentro de este subreddit ──
    # tqdm() envuelve la lista para mostrar una barra de progreso
    for keyword in tqdm(ALL_KEYWORDS, desc=f"  Keywords", leave=False):
        try:
            # Hacer la búsqueda en Reddit
            # subreddit.search() devuelve una lista de posts que contienen la keyword
            # sort="relevance" → ordena por qué tan relevante es el resultado
            # limit=50 → máximo 50 posts por búsqueda (límite de la API)
            resultados = list(subreddit.search(
                keyword,
                sort="relevance",
                limit=MAX_POSTS_POR_BUSQUEDA
            ))

            # ── PASO 3: Procesar cada post encontrado ──
            for post in resultados:

                # ¿Está dentro de nuestro período 2016-2026?
                # Si no, lo saltamos (la API no filtra por fecha)
                if not fecha_en_rango(post.created_utc):
                    continue

                # ¿Ya procesamos este post antes (con otra keyword)?
                # Si sí, lo saltamos para no tener duplicados
                if post.id in ids_vistos:
                    continue
                ids_vistos.add(post.id)  # Marcarlo como ya visto

                # Extraer los datos del post y agregarlo a nuestra lista
                post_data = extraer_post(post, keyword)
                if post_data:                # Si la extracción fue exitosa (no None)
                    datos.append(post_data)  # Agregar al final de la lista
                    count += 1               # Sumar 1 al contador de este subreddit

                # ── PASO 4: Extraer comentarios de este post ──
                # Los comentarios tienen el 92.6% del contenido del corpus
                try:
                    # replace_more(limit=0) le dice a PRAW que NO expanda
                    # los botones de "load more comments"
                    # Esto hace la extracción mucho más rápida
                    post.comments.replace_more(limit=0)

                    # .list() convierte el árbol de comentarios en lista plana
                    # [:20] toma solo los primeros 20
                    for comment in post.comments.list()[:MAX_COMENTARIOS_POR_POST]:

                        # Evitar comentarios duplicados (mismo control que con posts)
                        if comment.id in ids_vistos:
                            continue
                        ids_vistos.add(comment.id)

                        # Extraer datos del comentario
                        com_data = extraer_comentario(comment, sub_name, keyword)
                        if com_data:
                            datos.append(com_data)
                            count += 1

                except Exception:
                    # Si fallan los comentarios de un post específico,
                    # no es grave: simplemente seguimos con el siguiente post
                    pass

            # Pausa entre búsquedas para respetar el rate limit de Reddit
            # Sin esta pausa, Reddit bloquea temporalmente nuestras peticiones
            time.sleep(PAUSA_ENTRE_BUSQUEDAS)

        except prawcore.exceptions.TooManyRequests:
            # Reddit nos dice que hicimos demasiadas peticiones
            # Esperamos 60 segundos y seguimos automáticamente
            print("  Rate limit alcanzado — esperando 60s...")
            time.sleep(60)

        except Exception as e:
            # Cualquier otro error: lo guardamos en el log y seguimos
            errores.append({"subreddit": sub_name, "keyword": keyword, "error": str(e)})

    # Mostrar cuántos textos se extrajeron de este subreddit
    print(f"  Extraídos: {count:,}")  # :, agrega comas (ej: 1,234)

    # ── CHECKPOINT: guardar progreso cada N subreddits ──
    # Si la conexión se cae o hay un error, no perdemos todo el trabajo
    if idx % CHECKPOINT_CADA_N_SUBS == 0:  # % es módulo: da 0 cada N
        df_checkpoint = pd.DataFrame(datos)  # Convertir lista a tabla
        df_checkpoint.to_csv("../data/raw/reddit_data_parcial.csv", index=False)
        print(f"  Checkpoint guardado: {len(datos):,} registros")

# ── Resumen final de la extracción ───────────────────────
tiempo_total = time.time() - t_inicio  # Segundos totales
print(f"\nExtracción completada en {tiempo_total / 3600:.2f} horas")
print(f"Total de registros extraídos: {len(datos):,}")
print(f"Errores registrados: {len(errores)}")


[1/17] r/TwoXChromosomes


  Extraídos: 30,022

[2/17] r/girlsgonewired


  Extraídos: 8,165

[3/17] r/WomenInTech


  Extraídos: 14,968
  Checkpoint guardado: 53,155 registros

[4/17] r/LadiesofScience


  Extraídos: 4,637

[5/17] r/WomenEngineers


...

  Extraídos: 9,508

[14/17] r/MachineLearning


  Extraídos: 5,558

[15/17] r/technology


  Extraídos: 5,529
  Checkpoint guardado: 168,185 registros

[16/17] r/antiwork


  Extraídos: 24,582

[17/17] r/AskWomen


  Extraídos: 8,415

Extracción completada en 2.83 horas
Total de registros extraídos: 201,182
Errores registrados: 0


---
## 6. Guardar dataset final

Se generan dos archivos:
1. **CSV completo** con todos los datos (NO se sube a GitHub por privacidad)
2. **CSV de IDs** que SÍ se sube a GitHub (permite reconstruir el corpus)

In [38]:
# ══════════════════════════════════════════════════════════
# CREAR DATAFRAME Y ELIMINAR DUPLICADOS
# ══════════════════════════════════════════════════════════

# Convertir la lista de diccionarios a un DataFrame de pandas
# Es como pasar de una lista de fichas a una tabla de Excel
df_raw = pd.DataFrame(datos)

# Eliminar duplicados por ID
# Un post puede aparecer en varias búsquedas si coincide con más de una keyword
# keep="first" conserva la primera aparición y elimina las repetidas
antes = len(df_raw)  # Cuántos registros hay antes de deduplicar
df_raw = df_raw.drop_duplicates(subset=["id"], keep="first")
despues = len(df_raw)  # Cuántos quedan después

print(f"Registros antes de deduplicar : {antes:,}")
print(f"Duplicados eliminados         : {antes - despues:,}")
print(f"Registros únicos finales      : {despues:,}")

Registros antes de deduplicar : 201,182
Duplicados eliminados         : 0
Registros únicos finales      : 201,182


In [39]:
# ══════════════════════════════════════════════════════════
# GUARDAR ARCHIVOS
# ══════════════════════════════════════════════════════════

# ── Archivo 1: Dataset completo (NO se sube a GitHub) ────
# Se agrega timestamp al nombre para saber exactamente cuándo se generó
# Este archivo está protegido por .gitignore (data/raw/*.csv)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")  # Ej: "20260115_143022"
ruta_csv = f"../data/raw/reddit_data_raw_{ts}.csv"
df_raw.to_csv(ruta_csv, index=False, encoding="utf-8")  # index=False para no guardar el índice
print(f"Dataset completo guardado: {ruta_csv}")

# ── Archivo 2: Solo IDs públicos (SÍ se sube a GitHub) ───
# Este archivo contiene solo los IDs, subreddit, tipo y fecha
# NO contiene los textos, así que es seguro compartirlo
# Otro investigador puede usar estos IDs para reconstruir el corpus
# descargando los textos con la API de Reddit
ruta_ids = "../data/raw/reddit_ids_publicos.csv"
df_raw[["id", "subreddit", "tipo", "fecha"]].to_csv(ruta_ids, index=False)
print(f"IDs públicos guardados  : {ruta_ids}")

Dataset completo guardado: ../data/raw/reddit_data_raw_20260318_174918.csv
IDs públicos guardados  : ../data/raw/reddit_ids_publicos.csv


---
## 7. Resumen del notebook

In [40]:
# ══════════════════════════════════════════════════════════
# RESUMEN FINAL
# Imprime un resumen con toda la información relevante
# de lo que se hizo en este notebook
# ══════════════════════════════════════════════════════════

print("=" * 55)
print("RESUMEN — NOTEBOOK 01: EXTRACCIÓN")
print("=" * 55)
print(f"  Subreddits analizados : {len(ALL_SUBREDDITS)}")           # 17
print(f"  Keywords utilizadas   : {len(ALL_KEYWORDS)}")             # 45
print(f"  Textos extraídos      : {len(df_raw):,}")                 # Total con comas
print(f"  Posts                 : {(df_raw['tipo'] == 'post').sum():,}")   # Solo posts
print(f"  Comentarios           : {(df_raw['tipo'] == 'comment').sum():,}")# Solo comentarios
print(f"  Período               : {df_raw['año'].min()} – {df_raw['año'].max()}")  # Rango real
print(f"  Errores               : {len(errores)}")                  # Cuántos errores hubo
print(f"  Dataset completo      : {ruta_csv}")                      # Ruta del archivo
print(f"  IDs públicos          : {ruta_ids}")                      # Ruta de los IDs
print("=" * 55)
print("\n→ Siguiente paso: notebooks/02_Analisis_Exploratorio.ipynb")

RESUMEN — NOTEBOOK 01: EXTRACCIÓN
  Subreddits analizados : 17
  Keywords utilizadas   : 58
  Textos extraídos      : 201,182
  Posts                 : 13,971
  Comentarios           : 187,211
  Período               : 2016 – 2026
  Errores               : 0
  Dataset completo      : ../data/raw/reddit_data_raw_20260318_174918.csv
  IDs públicos          : ../data/raw/reddit_ids_publicos.csv

→ Siguiente paso: notebooks/02_Analisis_Exploratorio.ipynb
